In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from ucimlrepo import fetch_ucirepo
import zipfile
import sys
import os
from pathlib import Path

# --- KAGGLE AUTHENTICATION FIX ---
# Put your actual username and key from the kaggle.json file here:
os.environ['KAGGLE_USERNAME'] = "omarhossam6"
os.environ['KAGGLE_KEY'] = "KGAT_b8c682b9434ee0d411951549520f6bce"

# We must import kaggle AFTER setting the environment variables
import kaggle 
# ---------------------------------

# Add the src folder to the path so we can import our config
sys.path.append(str(Path.cwd().parent))
import src.config as config

# Setup visualization style
sns.set_theme(style="whitegrid")
print("Libraries imported and Kaggle authenticated successfully!")

2026-04-08 19:17:06 - src.config - INFO - Project configuration loaded and directories verified.


Libraries imported and Kaggle authenticated successfully!


In [2]:
print("Downloading UCI Datasets...")

# Dataset 1: Student Performance (Math & Portuguese)
student_perf = fetch_ucirepo(id=320)
df_student_math = student_perf.data.original
# Save to raw data folder
df_student_math.to_csv(config.RAW_DATA_DIR / "uci_student_performance.csv", index=False)

# Dataset 2: Dropout & Academic Success
dropout_success = fetch_ucirepo(id=697)
df_dropout = dropout_success.data.original
df_dropout.to_csv(config.RAW_DATA_DIR / "uci_dropout_success.csv", index=False)

print("UCI Datasets downloaded and saved to /data/raw/")

UCI Datasets downloaded and saved to /data/raw/


In [4]:
print("Downloading Kaggle Dataset...")
dataset_name = "rabieelkharoua/students-performance-dataset"
kaggle_download_path = config.RAW_DATA_DIR

# Download using the Kaggle API
kaggle.api.dataset_download_files(
    dataset_name, 
    path=kaggle_download_path, 
    unzip=True
)

# Rename the extracted file for consistency
kaggle_file = list(kaggle_download_path.glob("Student_performance_data _.csv"))[0]
df_kaggle = pd.read_csv(kaggle_file)
kaggle_file.rename(config.RAW_DATA_DIR / "kaggle_student_performance.csv")

print("Kaggle dataset downloaded and unzipped to /data/raw/")

Dataset URL: https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset
Kaggle dataset downloaded and unzipped to /data/raw/


In [5]:
datasets = {
    "UCI_Student_Performance": df_student_math,
    "UCI_Dropout_Success": df_dropout,
    "Kaggle_Student_Performance": df_kaggle
}

for name, df in datasets.items():
    print(f"Generating profile for {name}...")
    profile = ProfileReport(df, title=f"{name} Profiling Report", minimal=True)
    # Save reports to the paper folder or root for easy access
    report_path = config.PROJECT_ROOT / "paper" / f"{name}_EDA_Report.html"
    profile.to_file(report_path)

print("All HTML EDA reports generated successfully in the /paper directory!")

Generating profile for UCI_Student_Performance...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 33/33 [00:00<00:00, 127.31it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 15/15 [00:00<00:00, 55043.36it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

All HTML EDA reports generated successfully in the /paper directory!


In [6]:
def plot_class_distribution(df, target_col, dataset_name):
    """Plots the class distribution to check for imbalance."""
    if target_col in df.columns:
        plt.figure(figsize=(8, 4))
        sns.countplot(data=df, x=target_col, palette="viridis")
        plt.title(f"Class Distribution: {target_col} ({dataset_name})")
        plt.show()

def plot_missing_heatmap(df, dataset_name):
    """Plots a heatmap of missing values."""
    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(10, 5))
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
        plt.title(f"Missing Values Heatmap: {dataset_name}")
        plt.show()
    else:
        print(f"No missing values found in {dataset_name}!")

# Example Audit: UCI Dropout dataset target is 'Target'
plot_class_distribution(df_dropout, 'Target', 'UCI Dropout Success')
plot_missing_heatmap(df_dropout, 'UCI Dropout Success')

No missing values found in UCI Dropout Success!


C:\Users\Omar\AppData\Local\Temp\ipykernel_32888\932140533.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x=target_col, palette="viridis")
C:\Users\Omar\AppData\Local\Temp\ipykernel_32888\932140533.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def detect_outliers_iqr(df):
    """Returns a dictionary of column names and their outlier counts using the IQR method."""
    outliers = {}
    # Only calculate for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count how many values fall outside the bounds
        outlier_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
        if outlier_count > 0:
            outliers[col] = outlier_count
            
    return outliers

print("Outliers in Kaggle Dataset (IQR method):")
print(detect_outliers_iqr(df_kaggle))

Outliers in Kaggle Dataset (IQR method):
{'ParentalEducation': 120, 'Music': 471, 'Volunteering': 376}


In [8]:
# Create a unified list of features
feature_inventory = pd.DataFrame({
    'UCI_Student': pd.Series(df_student_math.columns),
    'UCI_Dropout': pd.Series(df_dropout.columns),
    'Kaggle_Student': pd.Series(df_kaggle.columns)
})

feature_inventory.to_csv(config.PROJECT_ROOT / "paper" / "feature_mapping_worksheet.csv", index=False)
print("Feature mapping worksheet exported to /paper/")

Feature mapping worksheet exported to /paper/
